# Backtester Analysis & Validation
This notebook runs the backtester on a single day of LOB data and visualises:
- Equity Curve (Total / Realized / Unrealized PnL)
- Greek Exposures Over Time (Delta, Gamma, Vega, Theta)
- Fill Rate & Trade Analysis
- PnL Attribution (Theta, Gamma, Vega, Spread)
- Risk Breach Events
- Summary Statistics

# Import Libraries

In [98]:
# Reloading extension
%load_ext autoreload
# Reloads all modules every time before executing the Python code
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [99]:
import sys
from pathlib import Path
import logging
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

logging.basicConfig(level=logging.INFO)

from src.pricing_engine import PricingEngine
from src.quoting.lucic_tse import LucicTseQuotingEngine
from src.risk.risk_manager import RiskManager
from src.backtest.historical_connector import HistoricalConnector
from src.backtest.backtest_pricing_engine import HistoricalPricingEngine
from src.backtest.engine import Backtester
from src.data.option_spec import OptionSpec
from src.data.lob_snapshot import LOBSnapshot

# Load Historical Data & Extract State

## Load file

In [100]:
SYMBOL = "SPY"
R = 0.045
Q = 0.012
DATA_PATH = Path("../data/lob_history/SPY_2026-07-15.parquet")
INITIAL_CAPITAL = 100000.0

if not DATA_PATH.exists():
    print(f"❌ Data file not found: {DATA_PATH}")
    print("Please run collect_lob_data.py first.")
    raise FileNotFoundError("LOB data required")

# 1. Load the historical LOB data
lob_df = pd.read_parquet(DATA_PATH)
lob_df['timestamp'] = pd.to_datetime(lob_df['timestamp'])
lob_df = lob_df.sort_values('timestamp').reset_index(drop=True)

# 2. Extract the first timestamp and historical spot
first_timestamp = lob_df['timestamp'].iloc[0]
historical_spot = lob_df['spot_price'].iloc[0] # Spot price may move throughout the day

# 3. Get the expiries available in this historical data
historical_connector = HistoricalConnector(lob_df)
historical_expiries = historical_connector.get_available_expiries() # Initial timestamp

print(f"📅 Historical Date: {first_timestamp.date()}")
print(f"💰 Historical (market open) Spot: {historical_spot:.2f}")
print(f"📆 Expiries available: {historical_expiries}")

📅 Historical Date: 2026-07-15
💰 Historical (market open) Spot: 753.13
📆 Expiries available: ['2026-07-15', '2027-01-15', '2026-12-31', '2027-03-31', '2027-03-19', '2026-11-20', '2026-10-30', '2026-11-30', '2026-10-16', '2026-12-18', '2028-01-21', '2028-06-16', '2027-12-17', '2028-12-15', '2027-06-17', '2027-06-30', '2027-09-17', '2026-07-31', '2026-07-24', '2026-08-07', '2026-08-14', '2026-07-17', '2026-07-20', '2026-07-16', '2026-07-22', '2026-07-21', '2026-07-23', '2026-09-18', '2026-09-30', '2026-08-21', '2026-08-28', '2026-08-31']


## 2. Build PricingEngine (Calibrated to Historical Data)

In [101]:
# Inspect data
expiries = historical_connector.get_available_expiries()

raw_chains = {}
for exp in expiries:
    raw_chains[exp] = historical_connector.get_chain_for_expiry(exp, use_cache=False)
raw_chains

{'2026-07-15': {'calls': [OptionQuote(strike=675.0, bid=76.99, ask=79.8, mid=78.395, implied_vol=1.0419969775390627, volume=0, open_interest=7, option_type='call'),
   OptionQuote(strike=727.0, bid=24.96, ask=27.7, mid=26.33, implied_vol=0.6123085644531252, volume=11, open_interest=14, option_type='call'),
   OptionQuote(strike=728.0, bid=25.12, ask=26.69, mid=25.905, implied_vol=0.594730615234375, volume=0, open_interest=4, option_type='call'),
   OptionQuote(strike=729.0, bid=23.04, ask=25.85, mid=24.445, implied_vol=0.5961954443359375, volume=1, open_interest=1, option_type='call'),
   OptionQuote(strike=730.0, bid=21.89, ask=24.5, mid=23.195, implied_vol=0.5393112475585939, volume=31, open_interest=33, option_type='call'),
   OptionQuote(strike=731.0, bid=20.88, ask=23.55, mid=22.215, implied_vol=0.5285691674804689, volume=47, open_interest=49, option_type='call'),
   OptionQuote(strike=732.0, bid=19.94, ask=22.63, mid=21.285, implied_vol=0.5210008837890624, volume=2, open_interest

In [102]:
# 1. Create PricingEngine (without auto-calibration)
pricing_engine = HistoricalPricingEngine(
    initial_timestamp=first_timestamp,
    symbol=SYMBOL,
    r=R,
    q=Q,
    max_expiries=4,
    cache=False,  # Prevent live fetching
    connector=historical_connector # Used for backtesting
)

# 3. Calibrate using historical data
print("Calibrating PricingEngine to historical data...")
pricing_engine._calibrate()
print("✅ PricingEngine calibrated to historical data.")

# Get the calibrated SVI slices (which are now historical)
svi_slices = pricing_engine.get_svi_params()


Calibrating PricingEngine to historical data...
✅ PricingEngine calibrated to historical data.


## Build Option Specs

In [103]:
# Choose expiries equal to those calibrated
svi_slices = pricing_engine.get_svi_params()
expiries = sorted(svi_slices.keys())[:2]

spot = historical_connector.get_spot_price()

# Construct option specs around ATM
option_specs = []
for expiry in expiries:
    T = svi_slices[expiry]['T']
    strikes = [int(spot * (1 + i * 0.01)) for i in range(-3, 4)]
    for strike in strikes:
        option_specs.append(OptionSpec(
            strike=float(strike),
            expiry=expiry,
            T=T,
            option_type='call',
        ))
print(f"Created {len(option_specs)} option specs.")

Created 14 option specs.


## Setup QuotingEngine

In [104]:
risk_factors = [
    {
        'name': 'Vega 0-7D',
        'alpha': 10.0,
        'membership': lambda spec: spec.T <= 7/365,
        'weight_key': 'vega',
    },
    {
        'name': 'Vega 8-30D',
        'alpha': 5.0,
        'membership': lambda spec: 7/365 < spec.T <= 30/365,
        'weight_key': 'vega',
    },
    {
        'name': 'Total Delta',
        'alpha': 1.0,
        'membership': lambda spec: True,
        'weight_key': 'delta',
    },
]
order_flow_params = {
    'lambda0_a': 50 * 252,
    'lambda0_b': 50 * 252,
    'kappa_a': 0.75,
    'kappa_b': 0.75,
}

# The quoting engine will use the historically calibrated PricingEngine
quoting_engine = LucicTseQuotingEngine(
    pricing_engine=pricing_engine,
    risk_factors=risk_factors,
    order_flow_params=order_flow_params,
    horizon_hours=0.5,
    risk_aversion=0.1,
    auto_update=False,  # Disable background updates to avoid mixing with historical data
    initial_realized_vol=0.18,
    option_specs=option_specs,
)

# Update (Initialize)
quoting_engine.update_state()


print("Waiting for quoting engine initialization...")
while not quoting_engine.is_initialized():
    time.sleep(0.5)
print("✅ Quoting engine initialized.")

Waiting for quoting engine initialization...
✅ Quoting engine initialized.


## Setup Risk Manager

In [105]:
risk_manager = RiskManager(
    quoting_engine=quoting_engine,
    delta_limit=50000.0,
    gamma_limit=-100.0,
    theta_limit=-100.0,
    vega_limits={"0-7D": 500.0, "8-30D": 1500.0},
    drawdown_limit=-0.02,
    initial_capital=INITIAL_CAPITAL,
)
print("✅ Risk manager initialized.")


✅ Risk manager initialized.


## Setup Backtester

In [106]:
backtester = Backtester(
    pricing_engine=pricing_engine,
    quoting_engine=quoting_engine,
    risk_manager=risk_manager,
    option_specs=option_specs,
    data_path=DATA_PATH,
    initial_capital=INITIAL_CAPITAL,
)

# Run Backtester

## Run Backtester

In [107]:
results = backtester.run()

# Summary & Results

## Results

In [108]:
print("\n" + "="*60)
print("BACKTEST RESULTS")
print("="*60)
print(f"Total PnL:       ${results['total_pnl']:.2f}")
print(f"Sharpe Ratio:    {results['sharpe_ratio']:.2f}")
print(f"Max Drawdown:    {results['max_drawdown']:.2%}")
print(f"Win Rate:        {results['win_rate']:.2%}")
print(f"Total Trades:    {results['total_trades']}")
print(f"Risk Breaches:   {results['risk_breaches']}")
print(f"Drawdown Breaches: {results['drawdown_breaches']}")


BACKTEST RESULTS
Total PnL:       $-100101.02
Sharpe Ratio:    66.16
Max Drawdown:    -1041.73%
Win Rate:        0.00%
Total Trades:    66
Risk Breaches:   34
Drawdown Breaches: 31


## Plots

In [109]:
equity = results['equity_curve']

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=("Total PnL", "Realized vs Unrealized PnL", "Daily PnL Change"),
                    vertical_spacing=0.10)

# Total PnL
fig.add_trace(
    go.Scatter(x=equity['timestamp'], y=equity['total_pnl'],
               mode='lines', name='Total PnL', line=dict(color='blue', width=2)),
    row=1, col=1
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)

# Realized vs Unrealized
fig.add_trace(
    go.Scatter(x=equity['timestamp'], y=equity['realized_pnl'],
               mode='lines', name='Realized PnL', line=dict(color='green', width=1.5)),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=equity['timestamp'], y=equity['unrealized_pnl'],
               mode='lines', name='Unrealized PnL', line=dict(color='orange', width=1.5)),
    row=2, col=1
)

# Daily PnL Change
pnl_change = equity['total_pnl'].diff().fillna(0)
fig.add_trace(
    go.Bar(x=equity['timestamp'], y=pnl_change, name='PnL Change'),
    row=3, col=1
)

fig.update_layout(height=900, title_text="Equity Curve", showlegend=True)
fig.show()

## Greeks

In [110]:
# Extract Greeks from inventory history
greeks_df = pd.DataFrame([{
    'timestamp': s.timestamp,
    'delta': s.delta,
    'gamma': s.gamma,
    'vega': s.vega,
    'theta': s.theta,
} for s in backtester._inventory_history])

fig = make_subplots(rows=2, cols=2, subplot_titles=("Delta", "Gamma", "Vega", "Theta"))

fig.add_trace(go.Scatter(x=greeks_df['timestamp'], y=greeks_df['delta'],
                         mode='lines', name='Delta', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=greeks_df['timestamp'], y=greeks_df['gamma'],
                         mode='lines', name='Gamma', line=dict(color='red')), row=1, col=2)
fig.add_trace(go.Scatter(x=greeks_df['timestamp'], y=greeks_df['vega'],
                         mode='lines', name='Vega', line=dict(color='green')), row=2, col=1)
fig.add_trace(go.Scatter(x=greeks_df['timestamp'], y=greeks_df['theta'],
                         mode='lines', name='Theta', line=dict(color='orange')), row=2, col=2)

fig.update_layout(height=600, title_text="Greek Exposures Over Time")
fig.show()

## Fills

In [111]:
fills_df = pd.DataFrame([{
    'timestamp': f.timestamp,
    'option_id': f.option_id,
    'side': f.side,
    'price': f.price,
    'quantity': f.quantity,
    'notional': f.price * f.quantity,
} for f in backtester._fill_history])

if not fills_df.empty:
    total_fills = len(fills_df)
    buys = len(fills_df[fills_df['side'] == 'BUY'])
    sells = len(fills_df[fills_df['side'] == 'SELL'])
    total_notional = fills_df['notional'].sum()
    avg_trade_size = fills_df['notional'].mean()

    print(f"Total Fills: {total_fills}")
    print(f"Buys: {buys} ({buys/total_fills*100:.1f}%)")
    print(f"Sells: {sells} ({sells/total_fills*100:.1f}%)")
    print(f"Total Notional Traded: ${total_notional:,.2f}")
    print(f"Avg Trade Size: ${avg_trade_size:,.2f}")

    # Plot fill volume over time
    fill_counts = fills_df.groupby(fills_df['timestamp'].dt.floor('5min')).size()
    
    fig = go.Figure()
    fig.add_trace(go.Bar(x=fill_counts.index, y=fill_counts.values, name='Fills per 5min'))
    fig.update_layout(title="Fill Volume Over Time", xaxis_title="Time", yaxis_title="Number of Fills")
    fig.show()
else:
    print("No fills recorded in this backtest.")

Total Fills: 66
Buys: 57 (86.4%)
Sells: 9 (13.6%)
Total Notional Traded: $2,052.92
Avg Trade Size: $31.10


## PnL

In [112]:
if len(greeks_df) > 1:
    greeks_df['dt'] = greeks_df['timestamp'].diff().dt.total_seconds() / (60 * 60 * 24 * 365)
    greeks_df['dt'] = greeks_df['dt'].fillna(0)

    # Theta PnL: theta * dt
    greeks_df['theta_pnl'] = greeks_df['theta'] * greeks_df['dt'] * 252  # annualize

    # Gamma PnL: 0.5 * gamma * S^2 * (dS/S)^2
    # We need spot prices for this. We can approximate from the equity curve.
    # For simplicity, we'll just show the greeks.
    pass

# Plot attribution (simple version)
fig = make_subplots(rows=2, cols=1, subplot_titles=("Cumulative Realized PnL", "Inventory Value"))

# Cumulative realized PnL from fills
if not fills_df.empty:
    cum_pnl = fills_df.groupby('timestamp')['notional'].cumsum().reset_index(drop=True)
    fig.add_trace(
        go.Scatter(x=fills_df['timestamp'], y=cum_pnl,
                   mode='lines', name='Cumulative Realized PnL'),
        row=1, col=1
    )

# Inventory value over time (from MTM)
fig.add_trace(
    go.Scatter(x=equity['timestamp'], y=equity['unrealized_pnl'],
               mode='lines', name='Inventory Value (Unrealized PnL)', line=dict(color='orange')),
    row=2, col=1
)

fig.update_layout(height=600, title_text="PnL Attribution")
fig.show()


## Risk-Breaches

In [113]:
if results['risk_breaches'] > 0:
    print(f"⚠️ {results['risk_breaches']} risk breaches occurred.")
    print(f"   - {results['drawdown_breaches']} were drawdown breaches.")
    
    # Find timestamps of risk breaches from inventory history
    # We can infer breaches when Greeks hit limits
    fig = go.Figure()
    
    # Plot Delta with limit lines
    fig.add_trace(go.Scatter(x=greeks_df['timestamp'], y=greeks_df['delta'],
                             mode='lines', name='Delta', line=dict(color='blue')))
    fig.add_hline(y=50000, line_dash="dash", line_color="red", annotation_text="Delta Limit +")
    fig.add_hline(y=-50000, line_dash="dash", line_color="red", annotation_text="Delta Limit -")
    
    fig.update_layout(title="Delta Exposure with Risk Limits",
                      xaxis_title="Time", yaxis_title="Delta")
    fig.show()
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=greeks_df['timestamp'], y=greeks_df['gamma'],
                             mode='lines', name='Gamma', line=dict(color='red')))
    fig.add_hline(y=-100, line_dash="dash", line_color="red", annotation_text="Gamma Limit")
    fig.update_layout(title="Gamma Exposure with Risk Limits",
                      xaxis_title="Time", yaxis_title="Gamma")
    fig.show()
else:
    print("✅ No risk breaches occurred. Risk limits are well-calibrated.")


⚠️ 34 risk breaches occurred.
   - 31 were drawdown breaches.


## Heatmaps

In [114]:
# Which options were traded most?
if not fills_df.empty:
    option_activity = fills_df.groupby('option_id').size().sort_values(ascending=False)
    
    fig = go.Figure()
    fig.add_trace(go.Bar(x=option_activity.index, y=option_activity.values))
    fig.update_layout(title="Trade Volume by Option",
                      xaxis_title="Option ID", yaxis_title="Number of Trades")
    fig.show()